In [1]:
spark

In [2]:
username = spark.sparkContext.sparkUser()
print(username)

catarinap


In [3]:
output_directory = f'/user/{username}/notebooks'
print(output_directory)

/user/catarinap/notebooks


In [4]:
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.window import Window
from pyspark.sql import DataFrame, SparkSession
import os

In [5]:
def create_queue_mapping_salesforce_df(spark: SparkSession):
    schema = T.StructType([
        T.StructField("queue", T.StringType(), nullable=False),
        T.StructField("team", T.StringType(), nullable=False),
        T.StructField("subdomain", T.StringType(), nullable=False),
        T.StructField("domain", T.StringType(), nullable=False),
        T.StructField("department", T.StringType(), nullable=False)
    ])
    
    data = [
        ('AML', 'Anti Money Laundering', 'Anti Money Laundering', 'Merchant Operations', 'Operations'),
        ('AntiMoneyLaunderingAML', 'Anti Money Laundering', 'Anti Money Laundering', 'Merchant Operations', 'Operations'),
        ('CustomerRiskReview', 'Customer Risk Review', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('CustomerRiskReviewUnclassified', 'Customer Risk Review', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('PlatformsKYCOperations', 'KYC Operations', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('Onboarding', 'Onboarding', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('OnboardingUnclassified', 'Onboarding', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('Underwriting', 'Underwriting', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('UnderwritingUnclassified', 'Underwriting', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('CreditRisk', 'Credit Risk', 'Merchant Risk', 'Merchant Operations', 'Operations'),
        ('MerchantFraud', 'Merchant Fraud', 'Merchant Risk', 'Merchant Operations', 'Operations'),
        ('MerchantFraudUnclassified', 'Merchant Fraud', 'Merchant Risk', 'Merchant Operations', 'Operations'),
        ('MerchantPotentialLiabilityMPL', 'Merchant Potential Liability', 'Merchant Risk', 'Merchant Operations', 'Operations'),
        ('MPLUnclassified', 'Merchant Potential Liability', 'Merchant Risk', 'Merchant Operations', 'Operations'),
        ('PaymentCardIndustryOperationsPCI', 'Payment Card Industry Operations', 'Scheme Operations', 'Merchant Operations', 'Operations'),
        ('PCI', 'Payment Card Industry Operations', 'Scheme Operations', 'Merchant Operations', 'Operations'),
        ('PCIOperationsUnclassified', 'Payment Card Industry Operations', 'Scheme Operations', 'Merchant Operations', 'Operations'),
        ('Scheme Operations', 'Scheme Operations', 'Scheme Operations', 'Merchant Operations', 'Operations'),
        ('SchemeOperations', 'Scheme Operations', 'Scheme Operations', 'Merchant Operations', 'Operations'),
        ('SchemeOperationsUnclassified', 'Scheme Operations', 'Scheme Operations', 'Merchant Operations', 'Operations'),
        ('AccountManagementPool', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('AcquiringJapan', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Commercial_Tools_Support', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('FinanceQueue', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Impact', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('InternalSupportCasesQueue', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Lead Qualification', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('LeadQualification', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('LPM', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Nordics_Cases', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('PooledPartnerManagement', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Privacy', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Service_Case', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Risk', 'Risk', 'Risk', 'Professional Services', 'Operations'),
        ('RiskUnclassified', 'Risk', 'Risk', 'Professional Services', 'Operations'),
        ('Warehousing & Logistics', 'Warehousing & Logistics', 'Supply Chain', 'Supply Chain', 'Operations'),
        ('WarehousingLogistics', 'Warehousing & Logistics', 'Supply Chain', 'Supply Chain', 'Operations'),
        ('ComplaintsSupport', 'Operational Support', 'Operational Support', 'Support', 'Operations'),
        ('FranchiseesSupportFLS', 'Operational Support', 'Operational Support', 'Support', 'Operations'),
        ('OperationalSupport', 'Operational Support', 'Operational Support', 'Support', 'Operations'),
        ('PlatformMonitoring', 'Platform Monitoring', 'Platform Operations', 'Support', 'Operations'),
        ('PlatformMonitoringUnclassified', 'Platform Monitoring', 'Platform Operations', 'Support', 'Operations'),
        ('SupportUnclassified', 'Support Unclassified', 'Support Unclassified', 'Support', 'Operations'),
        ('AccountSetupOperationsGeneralists', 'Account Setup Operations', 'Technical Support', 'Support', 'Operations'),
        ('AccountSetupOperationsSpecialists', 'Account Setup Operations', 'Technical Support', 'Support', 'Operations'),
        ('Disputes', 'Disputes', 'Technical Support', 'Support', 'Operations'),
        ('DisputesUnclassified', 'Disputes', 'Technical Support', 'Support', 'Operations'),
        ('PartnerSupport', 'Technical Support', 'Technical Support', 'Support', 'Operations'),
        ('TechnicalSupportGeneralists', 'Technical Support', 'Technical Support', 'Support', 'Operations'),
        ('TechnicalSupportSpecialists', 'Technical Support', 'Technical Support', 'Support', 'Operations'),
        ('Account Management Pool', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Account Setup Operations', 'Account Setup Operations', 'Technical Support', 'Support', 'Operations'),
        ('Acquiring Japan', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Anti Money Laundering (AML)', 'Anti Money Laundering', 'Anti Money Laundering', 'Merchant Operations', 'Operations'),
        ('Claims Support', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Commercial Staging', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Commercial Tools Support', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Complaints Support', 'Operational Support', 'Operational Support', 'Support', 'Operations'),
        ('Credit Risk', 'Credit Risk', 'Merchant Risk', 'Merchant Operations', 'Operations'),
        ('Customer Risk Review', 'Customer Risk Review', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('Customer Risk Review Unclassified', 'Customer Risk Review', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('Disputes Unclassified', 'Disputes', 'Technical Support', 'Support', 'Operations'),
        ('Finance Queue', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Franchisees Support (FLS)', 'Operational Support', 'Operational Support', 'Support', 'Operations'),
        ('Internal Support Cases Queue', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Lead Qualification Unclassified', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Legal', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Merchant Fraud', 'Merchant Fraud', 'Merchant Risk', 'Merchant Operations', 'Operations'),
        ('Merchant Fraud Unclassified', 'Merchant Fraud', 'Merchant Risk', 'Merchant Operations', 'Operations'),
        ('Merchant Potential Liability (MPL)', 'Merchant Potential Liability', 'Merchant Risk', 'Merchant Operations', 'Operations'),
        ('MPL Unclassified', 'Merchant Potential Liability', 'Merchant Risk', 'Merchant Operations', 'Operations'),
        ('Nordics Cases', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Onboarding Unclassified', 'Onboarding', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('Operational Support', 'Operational Support', 'Operational Support', 'Support', 'Operations'),
        ('Partner Support', 'Technical Support', 'Technical Support', 'Support', 'Operations'),
        ('Payment Card Industry Operations (PCI)', 'Payment Card Industry Operations', 'Scheme Operations', 'Merchant Operations', 'Operations'),
        ('Payment Methods', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('PCI Operations Unclassified', 'Payment Card Industry Operations', 'Scheme Operations', 'Merchant Operations', 'Operations'),
        ('Platform Monitoring', 'Platform Monitoring', 'Platform Operations', 'Support', 'Operations'),
        ('Platform Monitoring Unclassified', 'Platform Monitoring', 'Platform Operations', 'Support', 'Operations'),
        ('Platforms KYC Operations', 'KYC Operations', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('Pooled Partner Management', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Risk Unclassified', 'Risk', 'Risk', 'Professional Services', 'Operations'),
        ('Scaled Partner Management', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Scheme Operations Unclassified', 'Scheme Operations', 'Scheme Operations', 'Merchant Operations', 'Operations'),
        ('Service Case', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Support Unclassified', 'Support Unclassified', 'Support Unclassified', 'Support', 'Operations'),
        ('Technical Support', 'Technical Support', 'Technical Support', 'Support', 'Operations'),
        ('Underwriting Unclassified', 'Underwriting', 'Customer Due Diligence', 'Merchant Operations', 'Operations')
    ]
    return spark.createDataFrame(data=data, schema=schema)

In [6]:
queue_mapping_df = create_queue_mapping_salesforce_df(spark)
queue_mapping_df.show(5)

AML,Anti Money Laundering,Anti Money Laundering,Merchant Operations,Operations
AntiMoneyLaunderingAML,Anti Money Laundering,Anti Money Laundering,Merchant Operations,Operations
CustomerRiskReview,Customer Risk Review,Customer Due Diligence,Merchant Operations,Operations
CustomerRiskReviewUnclassified,Customer Risk Review,Customer Due Diligence,Merchant Operations,Operations
PlatformsKYCOperations,KYC Operations,Customer Due Diligence,Merchant Operations,Operations


In [7]:
def add_support_team_column(df):
    
    support_team_col = (
        F.when(
            (F.col('language_c') == 'ja') & 
                ((F.col('team') == 'Operational Support') | (F.col('team') == 'Account Setup Operations'))
            , 'Japan - Operational Support'
        ).when(
            (F.col('language_c') == 'ja') & 
                (F.col('team') == 'Disputes')
            , 'Japan - Disputes'
        ).when(
            (F.col('language_c') == 'ja') & 
                (F.col('team') == 'Technical Support')
            , 'Japan - Technical Support'
        ).when(
            (F.col('language_c') == 'zh'), 'China'
        ).when(
            F.col('team') == 'Account Setup Operations', 'Account Setup Operations'
        ).when(
            (F.col('team') == 'Operational Support') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ), 'Operational Support'
        ).when(
            (F.col('team') == 'Disputes') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ), 'Disputes'
        ).when(
            (F.col('solution_focus_c') == 'Ecom APMs') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ) & (
                (F.col('language_c') != 'ja') & (F.col('language_c') != 'zh') | (F.col('language_c').isNull())
            ), 'Ecom APM'
        ).when(
            (F.col('solution_focus_c') == 'Ecom Cards') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ) & (
                (F.col('language_c') != 'ja') & (F.col('language_c') != 'zh') | (F.col('language_c').isNull())
            ), 'Ecom Cards'
        ).when(
            (F.col('solution_focus_c') == 'Ecom Core') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ) & (
                (F.col('language_c') != 'ja') & (F.col('language_c') != 'zh') | (F.col('language_c').isNull())
            ), 'Ecom Core'
        ).when(
            (F.col('solution_focus_c') == 'Ecom Integration') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ) & (
                (F.col('language_c') != 'ja') & (F.col('language_c') != 'zh') | (F.col('language_c').isNull())
            ), 'Ecom Integration'
        ).when(
            (F.col('solution_focus_c') == 'Finance') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ) & (
                (F.col('language_c') != 'ja') & (F.col('language_c') != 'zh') | (F.col('language_c').isNull())
            ), 'Finance'
        ).when(
            (F.col('solution_focus_c') == 'IPP Adv. Payments') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ) & (
                (F.col('language_c') != 'ja') & (F.col('language_c') != 'zh') | (F.col('language_c').isNull())
            ), 'IPP Adv. Payments'
        ).when(
            (F.col('solution_focus_c') == 'IPP Core') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ) & (
                (F.col('language_c') != 'ja') & (F.col('language_c') != 'zh') | (F.col('language_c').isNull())
            ), 'IPP Core'
        ).when(
            (F.col('solution_focus_c') == 'IPP Integration') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ) & (
                (F.col('language_c') != 'ja') & (F.col('language_c') != 'zh') | (F.col('language_c').isNull())
            ), 'IPP Integrations'
        ).when(
            (F.col('solution_focus_c') == 'Platforms') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ) & (
                (F.col('language_c') != 'ja') & (F.col('language_c') != 'zh') | (F.col('language_c').isNull())
            ), 'Platforms'
        ).when(
            (F.col('solution_focus_c') == 'Partner Support') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ) & (
                (F.col('language_c') != 'ja') & (F.col('language_c') != 'zh') | (F.col('language_c').isNull())
            ), 'Partner Support'
        ).when(
            (F.col('case_origin_region_c') == 'LATAM') & (
                (F.col('team') == 'Disputes') | 
                (F.col('team') == 'Operational Support')
            ), 'LATAM - Operational Support'
        ).when(
            (F.col('case_origin_region_c') == 'LATAM') & (
                (F.col('team') == 'Technical Support')
            ), 'LATAM - Technical Support'
        ).when(
            (F.col('source_region') == 'india') & (
                (F.col('team') == 'Technical Support')
            ), 'India - Technical Support'
        ).
        otherwise(None)
    )
    
    return df.withColumn(
        'support_team',
        F.when(F.col('domain') == 'Support', support_team_col).otherwise(None)
    )


In [8]:
cases_df = spark.table("com_orch.ingestion_salesforce_cases_global")
cases_df.show(2)

06344490,None,None,2025-12-21 09:20:45,None,0053W000002tc0OQAQ,2025-12-21 09:20:45,2025-12-21,EUR,"Missing: Reason, Associated Account and Contact",75.0,None,False,False,500QD00000mujJpYAI,None,True,False,None,0053W000002tc0OQAQ,2025-12-21 09:20:47,None,None,None,0.0,False,Email,00G3W000001WN4PUAW,None,None,00G3W000001WN4PUAW,Medium,None,None,0123W000000L3aoQAC,None,N/A,02sQD00000g6QHRYA2,Completed,Chargebacks,None,1ec01891f557df8bfb889dfa7e609ebd255d1108e8c85776a8ef2a98810e4e5c,31ca1e45a3ff34e0b2bd8b685b83c4ae0a3034530c0fe25d152353bee1b6869e,2025-12-21 09:20:47,None,Payments,None,None,None,None,None,None,None,None,true,Blocked Email,Disputes Unclassified,DisputesUnclassified,None,Service Case,None,None,None,"<a href=""/00G3W000001WN4P"" target=""_self"">Disputes Unclassified</a>",None,None,True,None,None,EMEA,DisputesUnclassified,false,None,None,None,False,None,None,None,None,None,None,None,None,None,False,False,None,2026-05-19 11:28:21.032399,global,False,False,None
06343751,None,None,2025-12-21 07:07:11,None,0053W000002tc0OQAQ,2025-12-21 07:07:11,2025-12-21,EUR,"Missing: Type, Reason, Associated Account and Contact",60.0,None,False,False,500QD00000mujLOYAY,Financial Services,True,False,None,0053W000002tc0OQAQ,2025-12-21 07:07:13,None,None,None,0.0,False,Email,00G3W000001WN4SUAW,None,support@adyen.com,00G3W000001WN4SUAW,Medium,None,None,0123W000000L3aoQAC,None,None,02sQD00000g6TwoYAE,Completed,None,None,e23b3d598cff9ed9783ed768614256ee57716d4b7c50b2ffb5dce695df355ff5,None,2025-12-21 07:07:15,None,None,None,None,None,None,None,0012400000UMEDtAAP,None,None,true,OutOfOffice,Support Unclassified,SupportUnclassified,None,Service Case,None,None,None,"<a href=""/00G3W000001WN4S"" target=""_self"">Support Unclassified</a>",None,None,False,None,None,None,SupportUnclassified,false,None,Digital,None,False,None,None,None,None,None,None,None,None,None,False,False,None,2026-05-19 11:28:21.032399,global,False,False,None


In [9]:
sla_df = spark.table("com_orch.ingestion_salesforce_case_milestone")
sla_df.show(2)

555QD00001ji8DxYAI,500QD00000gQ3lFYAS,2025-09-24 16:44:02,2025-09-25 12:44:00,None,557QD000000E890YAC,False,True,2025-09-24 16:44:03,2025-09-24 16:44:03,0053W000002tc0OQAQ,2025-09-24 16:44:03,0053W000002tc0OQAQ,False,1200,20.0,0.83,00:00,00:00,0.0,None,None,None,237731:58,3962:11,165.09164900462963,01mQD0000005v8wYAA,None,None,None,None,None,None,2026-05-19 11:41:51.969257
555QD00001ji8DyYAI,500QD00000gQ3lFYAS,2025-09-24 16:44:02,2025-10-08 20:44:00,None,557QD000000E891YAC,False,True,2025-09-24 16:44:03,2025-09-24 16:44:03,0053W000002tc0OQAQ,2025-09-24 16:44:03,0053W000002tc0OQAQ,False,14400,240.0,10.0,00:00,00:00,0.0,None,None,None,224531:58,3742:11,155.9249824074074,01mQD0000005v8wYAA,None,None,None,None,None,None,2026-05-19 11:41:51.969257


In [10]:
measures_df = spark.table("com_orch.salesforcecasescalculatedmeasures")
measures_df.show(2)

None,5003W00000NiIwlQAF,None,None,None,None,None,None,None,None,None,None,0,0,False,361946,None,0,None,None,0,0,global
None,5003W00000OvfWKQAZ,None,None,None,None,None,None,None,None,None,None,0,0,False,261662,None,0,None,None,0,0,global


In [11]:
def process_cases(cases_df, queue_mapping_df, measures_df, sla_df):
    
    # Step 1: Join with queue mapping
    clean_cases = cases_df.join(queue_mapping_df, cases_df.previous_queue_c == queue_mapping_df.queue, how='left') \
                          .drop(queue_mapping_df.queue)

    # Step 2: Join with measures data
    mid_cases = clean_cases.join(measures_df, clean_cases.id == measures_df.case_id, how='left') \
                             .drop(measures_df.case_id) \
                             .drop(measures_df.source_region)   

    # Step 3: Join with milestone data
    final_cases = mid_cases.join(sla_df, mid_cases.id == sla_df.case_id, how='left') \
                            .drop(sla_df.case_id) \
                            .drop(sla_df.id) \
                            .drop(sla_df.created_date) \
                            .drop(sla_df.is_deleted)

    final_cases = add_support_team_column(final_cases)

    # Step 4: Create 'category' column
    final_cases = final_cases.withColumn(
        'category', 
        F.when(F.col('emails_sent') > 0, 'Service').otherwise('Internal')
    )

    # Step 5: Apply filtering conditions
    filtered_cases = final_cases.filter(
        (~F.col("is_deleted") | F.col("is_deleted").isNull()) &  
        (F.col("is_closed")) &
        (F.col("domain") == "Support") &
        (F.col("category") == "Service") &
        (F.col("team") != "Platform Monitoring") &
        (F.col("created_date") >= "2024-01-01") &
        (F.col("milestone_type_id") == "557QD000000E891YAC") &
        (~(F.col("status") == "Merged") | F.col("status").isNull()) &  
        (~F.col("auto_close_reason_c").isin(["Blocked Email", "Shopper Case", "Zendesk Transfer", "Invalid Case", "Case Automation", "OutOfOffice", "RedirectedToCustomerArea", "Undelivered","Complaints Auto-Reply"]) | 
         F.col("auto_close_reason_c").isNull())  
    )

    # Steap 6: Select relevant columns
    relevant_cols = [
        "case_number",
        "id",
        "created_date",
        "closed_date",
        "team",
        "support_team",
        "actual_elapsed_time_in_mins",
        "elapsed_time_in_mins"    ]

    return filtered_cases.select(*relevant_cols)

In [12]:
aggregated_cases_df = process_cases(cases_df, queue_mapping_df, measures_df, sla_df)
aggregated_cases_df.show(5)

03426305,500QD0000075aObYAI,2024-01-08 07:21:27,2024-01-09 21:27:09,Operational Support,Operational Support,2285,2285
03436849,500QD000007KOAJYA4,2024-01-10 12:59:29,2024-01-17 15:39:01,Operational Support,LATAM - Operational Support,10239,10239
03454088,500QD000007by64YAA,2024-01-16 15:40:34,2024-06-21 12:44:29,Account Setup Operations,Account Setup Operations,225843,225843
03475201,500QD000007xvsfYAA,2024-01-23 23:32:06,2024-03-13 23:41:26,Account Setup Operations,Account Setup Operations,51009,51009
03483161,500QD0000086KaaYAE,2024-01-26 09:30:20,2024-03-18 11:59:14,Technical Support,None,75028,75028


In [13]:
history_df = spark.table("com_orch.ingestion_salesforce_casehistory_backfill")
history_df.show(5)

017QD000007EnPwYAK,DynamicEnum,Completed,New,2023-10-11 00:00:05,2023-10-11,Status,False,500QD000002VoXFYA0,005QD000000Yj1JYAS,2023-10-11 04:00:00,global
017QD000007EmtlYAC,Text,None,None,2023-10-11 00:00:10,2023-10-11,created,False,500QD000002Vm8uYAC,0053W000002tc2eQAA,2023-10-11 04:00:00,global
017QD000007EmtnYAC,EntityId,00GQD00000049862AA,00G3W000001WN4SUAW,2023-10-11 00:00:10,2023-10-11,Owner,False,500QD000002Vm8uYAC,0053W000002tc2eQAA,2023-10-11 04:00:00,global
017QD000007EkyUYAS,DynamicEnum,Refunds,None,2023-10-11 00:00:11,2023-10-11,ServiceTopic__c,False,500QD000002VW5mYAG,005QD000000YjCcYAK,2023-10-11 04:00:00,global
017QD000007EkyTYAS,DynamicEnum,Payments,None,2023-10-11 00:00:11,2023-10-11,Type,False,500QD000002VW5mYAG,005QD000000YjCcYAK,2023-10-11 04:00:00,global


In [14]:
def count_completed(aggregated_cases_df, history_df):

    completions = history_df.filter(
        (F.col("field") == "Status") & 
        (F.col("new_value") == "Completed")
    ).groupBy("case_id").agg(F.count("*").alias("completion_count")) \
     .withColumnRenamed("case_id", "id")

    window_spec = Window.partitionBy("case_id").orderBy("created_date")

    history_ordered = history_df.filter(F.col("field") == "Status") \
        .withColumn("prev_status", F.lag("new_value").over(window_spec)) \
        .withColumn("next_status", F.lead("new_value").over(window_spec))

    completions_ordered = history_ordered.filter(
        (F.col("new_value") == "Completed") &
        (F.col("prev_status") == "Waiting for Merchant") &
        (F.col("next_status") == "In Progress")
    ).groupBy("case_id").agg(F.count("*").alias("completion_ordered_count")) \
     .withColumnRenamed("case_id", "id")

    join_df = aggregated_cases_df \
        .join(completions, ["id"], how='left') \
        .join(completions_ordered, ["id"], how='left')

    return join_df.fillna(0, subset=["completion_count", "completion_ordered_count"])

In [15]:
history_combined = count_completed(aggregated_cases_df, history_df)
history_combined.show(10)

500QD0000075aObYAI,03426305,2024-01-08 07:21:27,2024-01-09 21:27:09,Operational Support,Operational Support,2285,2285,2,0
500QD000007KOAJYA4,03436849,2024-01-10 12:59:29,2024-01-17 15:39:01,Operational Support,LATAM - Operational Support,10239,10239,1,0
500QD000007by64YAA,03454088,2024-01-16 15:40:34,2024-06-21 12:44:29,Account Setup Operations,Account Setup Operations,225843,225843,1,0
500QD000007xvsfYAA,03475201,2024-01-23 23:32:06,2024-03-13 23:41:26,Account Setup Operations,Account Setup Operations,51009,51009,1,0
500QD0000086KaaYAE,03483161,2024-01-26 09:30:20,2024-03-18 11:59:14,Technical Support,None,75028,75028,1,0
500QD0000096yqkYAA,03533514,2024-02-12 14:28:03,2024-03-04 20:34:08,Account Setup Operations,Account Setup Operations,30606,30606,2,0
500QD000009D96jYAC,03543863,2024-02-14 08:33:05,2024-03-06 02:02:51,Technical Support,Platforms,29849,29849,1,0
500QD000009KU9oYAG,03550976,2024-02-16 03:23:56,2024-03-05 14:51:18,Disputes,Disputes,26607,26607,2,0
500QD000009XZZpYAO,03561149,2024-02-20 16:33:22,2024-03-08 11:25:03,Technical Support,Ecom Cards,14081,18171,3,0
500QD000009c3MKYAY,03564771,2024-02-21 17:02:07,2024-03-14 13:29:45,Technical Support,Finance,21387,31467,1,0


In [16]:
def get_monthly_report(df):
    """
    Groups only the 2025 closed tickets by month using 'closed_date'.
    """
    
    # 1. Cast closed_date to timestamp
    # 2. Filter for only the year 2025
    # 3. Create month_period and aggregate
    
    monthly_2025 = df.filter(
        (F.col("closed_date").isNotNull()) & 
        (F.col("closed_date").cast("timestamp") >= "2026-01-01") &
        (F.col("support_team").like('LATAM%') | F.col("support_team").like('Japan%'))
    ) \
    .withColumn(
        "month_period", 
        F.date_format(F.col("closed_date").cast("timestamp"), "yyyy-MM")
    ) \
    .groupBy("month_period","support_team") \
    .agg(
        F.countDistinct("id").alias("total_tickets"),
        F.round(F.avg(F.col("actual_elapsed_time_in_mins")), 2).alias("avg_actual_elapsed_time_in_mins")
    ) \
    .orderBy("month_period")

    return monthly_2025

In [17]:
monthly_analytics = get_monthly_report(history_combined)
monthly_analytics.show()

2026-01,Japan - Disputes,140,1074.96
2026-01,LATAM - Technical Support,397,4203.58
2026-01,LATAM - Operational Support,920,2214.06
2026-01,Japan - Operational Support,90,25847.9
2026-01,Japan - Technical Support,73,4018.11
2026-02,LATAM - Operational Support,1094,2063.12
2026-02,Japan - Disputes,123,1844.33
2026-02,Japan - Operational Support,97,16412.01
2026-02,LATAM - Technical Support,419,3867.42
2026-02,Japan - Technical Support,94,5468.19
2026-03,Japan - Technical Support,92,9207.78


In [18]:
def get_yearly_closed_report(df):
    """
    Groups closed tickets by year (2025, 2026+) 
    to show high-level performance trends.
    """
    
    # 1. Filter for January 2025 onwards
    # 2. Extract Year (YYYY)
    # 3. Aggregate total tickets and average elapsed time
    
    yearly_stats = df.filter(
        (F.col("closed_date").isNotNull()) & 
        (F.col("closed_date").cast("timestamp") >= "2025-01-01")
    ) \
    .withColumn(
        "year_period", 
        F.year(F.col("closed_date").cast("timestamp"))
    ) \
    .groupBy("year_period") \
    .agg(
        F.countDistinct("id").alias("total_tickets"),
        F.round(F.avg(F.col("actual_elapsed_time_in_mins")), 2).alias("avg_actual_elapsed_time_in_mins")
    ) \
    .orderBy("year_period")

    return yearly_stats

In [19]:
yearly_summary = get_yearly_closed_report(history_combined)
yearly_summary.show()

2025,297977,4198.88
2026,114773,4018.38


In [20]:
# 1. Prepare month for the entire dataset (no initial filter)
summary_base_df = history_combined.withColumn("month", F.date_format("closed_date", "yyyy-MM"))

summary_filtered_df = summary_base_df.filter(F.col("month") >= "2025-01")

# 2. Aggregate everything in one pass
final_summary = summary_filtered_df.groupBy("month").agg(
    # Total volume of all cases
    F.count("*").alias("total_cases"),
    
    # Total cases that were reopened (completed > 1)
    F.sum(F.when(F.col("completion_count") > 1, 1).otherwise(0))
        .alias("total_reopen_cases"),
    
    # Reopens that followed the workflow at least once
    F.sum(F.when((F.col("completion_count") > 1) & (F.col("completion_ordered_count") > 0), 1).otherwise(0))
        .alias("total_reopen_cases_with_corner_case")
    ).withColumn(
        # The "Incorrect" count (skipped workflow)
        "total_reopen_cases_100pct_incorrect", 
        F.col("total_reopen_cases") - F.col("total_reopen_cases_with_corner_case")
    ).withColumn(
        # NEW: Overall Reopen Rate (Percentage of total volume that are reopens)
        "pct_total_reopened",
        F.round((F.col("total_reopen_cases") / F.col("total_cases")) * 100, 2)
    ).withColumn(
        # Percentage of reopens that were incorrect
        "pct_total_reopened_100pct_incorrect",
        F.round((F.col("total_reopen_cases_100pct_incorrect") / F.col("total_cases")) * 100, 2)
    ).orderBy("month")
    
final_summary.show()

2025-01,22596,6831,915,5916,30.23,26.18
2025-02,23289,7098,945,6153,30.48,26.42
2025-03,25880,7619,1049,6570,29.44,25.39
2025-04,26359,7924,1186,6738,30.06,25.56
2025-05,25704,7373,1069,6304,28.68,24.53
2025-06,23537,7447,844,6603,31.64,28.05
2025-07,27900,8882,961,7921,31.84,28.39
2025-08,23924,7200,936,6264,30.1,26.18
2025-09,24447,6820,956,5864,27.9,23.99
2025-10,26425,7405,1093,6312,28.02,23.89
2025-11,24029,6567,1032,5535,27.33,23.03


In [21]:
# 1. Prepare YEAR for the entire dataset
yearly_base_df = history_combined.withColumn("year", F.date_format("closed_date", "yyyy"))

# 2. Aggregate using your exact logic
yearly_summary = yearly_base_df.groupBy("year").agg(
    F.count("*").alias("total_cases"),
    
    F.sum(F.when(F.col("completion_count") > 1, 1).otherwise(0))
        .alias("total_reopen_cases"),
    
    F.sum(F.when((F.col("completion_count") > 1) & (F.col("completion_ordered_count") > 0), 1).otherwise(0))
        .alias("total_reopen_cases_with_corner_case")
).withColumn(
    "total_reopen_cases_100pct_incorrect", 
    F.col("total_reopen_cases") - F.col("total_reopen_cases_with_corner_case")
).withColumn(
    "pct_total_reopened",
    F.round((F.col("total_reopen_cases") / F.col("total_cases")) * 100, 2)
).withColumn(
    "pct_total_reopened_100pct_incorrect",
    F.round((F.col("total_reopen_cases_100pct_incorrect") / F.col("total_cases")) * 100, 2)
).orderBy("year")

yearly_summary.show()

2024,242760,74875,8961,65914,30.84,27.15
2025,297981,87534,12263,75271,29.38,25.26
2026,114773,26310,5038,21272,22.92,18.53


In [22]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timedelta

@F.udf("double")
def calculate_biz_mins_precise_udf(start, end):
    if start is None or end is None or start >= end:
        return 0.0
    
    total_seconds = (end - start).total_seconds()
    dead_seconds = 0.0
    current_date = start.date()
    
    while current_date <= end.date():
        day_start = datetime.combine(current_date, datetime.min.time())
        day_end = day_start + timedelta(days=1)
        overlap_start = max(start, day_start)
        overlap_end = min(end, day_end)
        
        if overlap_start < overlap_end:
            overlap_duration = (overlap_end - overlap_start).total_seconds()
            if current_date.month == 1 and current_date.day == 1:
                dead_seconds += overlap_duration
            elif current_date.weekday() in (5, 6):
                dead_seconds += overlap_duration
            elif current_date.weekday() == 0:
                # Monday 00:00 - 02:00
                m_start = max(overlap_start, day_start)
                m_end = min(overlap_end, day_start + timedelta(hours=2))
                if m_start < m_end:
                    dead_seconds += (m_end - m_start).total_seconds()
        current_date += timedelta(days=1)
    return float((total_seconds - dead_seconds) / 60.0)

def calculate_business_minutes(history_df, cases_df):
    # 1. Prepare Cases (ID here is the Ticket ID)
    case_starts = cases_df.select(
        F.col("id").alias("case_id_join"), 
        F.col("created_date").alias("open_time")
    )

    # 2. Filter History and Join
    # history.case_id links to cases.id
    base_df = history_df.filter((F.col("field") == "Status") & (F.year("created_date") >= 2024)) \
        .join(F.broadcast(case_starts), F.col("case_id") == F.col("case_id_join"), "left")

    # 3. Define Window by CASE_ID (Grouping all logs for one ticket)
    window_spec = Window.partitionBy("case_id").orderBy("created_date")

    # 4. Create the Stints
    stints = base_df.withColumn("rn", F.row_number().over(window_spec)) \
        .withColumn("stint_start", 
            # Row 1 starts at the Ticket Open Time
            F.when((F.col("rn") == 1) & (F.col("open_time").isNotNull()), F.col("open_time"))
             .otherwise(F.col("created_date"))
        ) \
        .withColumn("stint_end", F.lead("created_date").over(window_spec))

    # 5. Calculate Minutes
    result_df = stints.withColumn(
        "biz_mins", 
        calculate_biz_mins_precise_udf(F.col("stint_start"), F.col("stint_end"))
    )

    # 6. Final Pivot by CASE_ID
    return result_df.groupBy("case_id").pivot("new_value").agg(
        F.round(F.sum("biz_mins"), 2)
    ).fillna(0)

In [23]:
# JAPAN TEST

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timedelta

@F.udf("double")
def calculate_biz_mins_precise_udf(start, end):
    if start is None or end is None or start >= end:
        return 0.0
    
    total_seconds = (end - start).total_seconds()
    dead_seconds = 0.0
    current_date = start.date()
    
    while current_date <= end.date():
        day_start = datetime.combine(current_date, datetime.min.time())
        day_end = day_start + timedelta(days=1)
        overlap_start = max(start, day_start)
        overlap_end = min(end, day_end)
        
        if overlap_start < overlap_end:
            overlap_duration = (overlap_end - overlap_start).total_seconds()
            
            # 1. Full Dead Days (New Year's Day and Weekends)
            if current_date.month == 1 and current_date.day == 1:
                dead_seconds += overlap_duration
            elif current_date.weekday() in (5, 6):
                dead_seconds += overlap_duration
                
            # 2. Working Days: Subtract everything outside 01:00 - 11:00 CET
            else:
                # Pre-work dead time: 00:00 to 01:00 CET (Midnight to 8am Japan)
                pre_work_end = day_start + timedelta(hours=1)
                p_start = max(overlap_start, day_start)
                p_end = min(overlap_end, pre_work_end)
                if p_start < p_end:
                    dead_seconds += (p_end - p_start).total_seconds()
                
                # Post-work dead time: 11:00 to 24:00 CET (6pm to Midnight Japan)
                post_work_start = day_start + timedelta(hours=11)
                post_start = max(overlap_start, post_work_start)
                post_end = min(overlap_end, day_end)
                if post_start < post_end:
                    dead_seconds += (post_end - post_start).total_seconds()
                    
        current_date += timedelta(days=1)
    return float((total_seconds - dead_seconds) / 60.0)

def calculate_business_minutes(history_df, cases_df):
    # 1. Prepare Cases (ID here is the Ticket ID)
    case_starts = cases_df.select(
        F.col("id").alias("case_id_join"), 
        F.col("created_date").alias("open_time")
    )

    # 2. Filter History and Join
    base_df = history_df.filter((F.col("field") == "Status") & (F.year("created_date") >= 2024)) \
        .join(F.broadcast(case_starts), F.col("case_id") == F.col("case_id_join"), "left")

    # 3. Define Window by CASE_ID (Grouping all logs for one ticket)
    window_spec = Window.partitionBy("case_id").orderBy("created_date")

    # 4. Create the Stints
    stints = base_df.withColumn("rn", F.row_number().over(window_spec)) \
        .withColumn("stint_start", 
            # Row 1 starts at the Ticket Open Time
            F.when((F.col("rn") == 1) & (F.col("open_time").isNotNull()), F.col("open_time"))
             .otherwise(F.col("created_date"))
        ) \
        .withColumn("stint_end", F.lead("created_date").over(window_spec))

    # 5. Calculate Minutes
    result_df = stints.withColumn(
        "biz_mins", 
        calculate_biz_mins_precise_udf(F.col("stint_start"), F.col("stint_end"))
    )

    # 6. Final Pivot by CASE_ID
    return result_df.groupBy("case_id").pivot("new_value").agg(
        F.round(F.sum("biz_mins"), 2)
    ).fillna(0)

In [32]:
# BRAZIL TEST

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timedelta

@F.udf("double")
def calculate_biz_mins_precise_brazil_udf(start, end):
    if start is None or end is None or start >= end:
        return 0.0
    
    total_seconds = (end - start).total_seconds()
    dead_seconds = 0.0
    current_date = start.date()
    
    while current_date <= end.date():
        day_start = datetime.combine(current_date, datetime.min.time())
        day_end = day_start + timedelta(days=1)
        overlap_start = max(start, day_start)
        overlap_end = min(end, day_end)
        
        if overlap_start < overlap_end:
            overlap_duration = (overlap_end - overlap_start).total_seconds()
            
            # 1. Full Dead Days (New Year's Day and Weekends)
            if current_date.month == 1 and current_date.day == 1:
                dead_seconds += overlap_duration
            elif current_date.weekday() in (5, 6):
                dead_seconds += overlap_duration
                
            # 2. Working Days: Subtract everything outside 12:00 - 22:00 CET (8am - 6pm BRT)
            else:
                # Pre-work dead time: 00:00 to 12:00 CET (Midnight to 8am Brazil)
                pre_work_end = day_start + timedelta(hours=12)
                p_start = max(overlap_start, day_start)
                p_end = min(overlap_end, pre_work_end)
                if p_start < p_end:
                    dead_seconds += (p_end - p_start).total_seconds()
                
                # Post-work dead time: 22:00 to 24:00 CET (6pm to Midnight Brazil)
                post_work_start = day_start + timedelta(hours=22)
                post_start = max(overlap_start, post_work_start)
                post_end = min(overlap_end, day_end)
                if post_start < post_end:
                    dead_seconds += (post_end - post_start).total_seconds()
                    
        current_date += timedelta(days=1)
    return float((total_seconds - dead_seconds) / 60.0)

def calculate_business_minutes(history_df, cases_df):
    # 1. Prepare Cases
    case_starts = cases_df.select(
        F.col("id").alias("case_id_join"), 
        F.col("created_date").alias("open_time")
    )

    # 2. Filter History and Join
    base_df = history_df.filter((F.col("field") == "Status") & (F.year("created_date") >= 2024)) \
        .join(F.broadcast(case_starts), F.col("case_id") == F.col("case_id_join"), "left")

    # 3. Define Window by CASE_ID
    window_spec = Window.partitionBy("case_id").orderBy("created_date")

    # 4. Create the Stints
    stints = base_df.withColumn("rn", F.row_number().over(window_spec)) \
        .withColumn("stint_start", 
            F.when((F.col("rn") == 1) & (F.col("open_time").isNotNull()), F.col("open_time"))
             .otherwise(F.col("created_date"))
        ) \
        .withColumn("stint_end", F.lead("created_date").over(window_spec))

    # 5. Calculate Minutes using the Brazil UDF
    result_df = stints.withColumn(
        "biz_mins", 
        calculate_biz_mins_precise_brazil_udf(F.col("stint_start"), F.col("stint_end"))
    )

    # 6. Final Pivot by CASE_ID
    return result_df.groupBy("case_id").pivot("new_value").agg(
        F.round(F.sum("biz_mins"), 2)
    ).fillna(0)

In [33]:
status_durations = calculate_business_minutes(history_df,history_combined)
status_durations.show(10)

500QD0000020H9kYAE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
500QD00000738TeYAI,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
500QD0000073fb0YAA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
500QD000007I5LNYA0,0.0,0.0,0.43,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
500QD000007vRp5YAE,551.67,207.55,2289.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,267.88,0.0,0.0
500QD000008HTcAYAW,0.0,0.0,0.73,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
500QD000008Pd1fYAC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
500QD000008UxVyYAK,0.0,0.0,202.08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,292.33,0.0,0.0
500QD000008vfQUYAY,1.47,0.0,2233.62,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1984.33,0.0,0.0
500QD000009MfdqYAC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [34]:
final_report = history_combined.join(status_durations, history_combined.id == status_durations.case_id, how="left")

In [35]:
final_report.filter(F.col("completion_ordered_count") > 1).show(10)

500QD00000CgeDWYAZ,03713898,2024-04-10 12:55:18,2024-05-22 12:44:13,Account Setup Operations,Account Setup Operations,21565,42468,3,2,500QD00000CgeDWYAZ,554.8,0.0,2792.62,0.0,0.0,0.0,0.0,0.0,2140.82,0.0,12500.7,0.0,0.0
500QD00000Dmt1BYAR,03798478,2024-04-30 09:16:51,2024-06-24 22:32:21,Technical Support,Ecom APM,21149,55995,3,2,500QD00000Dmt1BYAR,4200.0,2129.52,8528.38,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9142.1,0.0,0.0
500QD00000LoKdpYAF,04262027,2024-09-19 13:52:37,2024-10-22 09:00:06,Technical Support,Ecom Integration,5304,32227,3,2,500QD00000LoKdpYAF,2725.2,0.0,1801.55,0.0,0.0,0.0,0.0,0.0,285.4,0.0,8875.23,0.0,0.0
500QD00000NtMcHYAV,04383981,2024-10-28 10:02:06,2024-11-18 02:13:31,Technical Support,Ecom APM,3822,13677,5,2,500QD00000NtMcHYAV,1189.35,0.0,1810.65,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6000.0,0.0,0.0
500QD00000V5e4gYAB,04815500,2025-03-11 15:29:52,2025-03-25 03:45:22,Technical Support,Ecom Cards,3465,13455,3,2,500QD00000V5e4gYAB,1290.2,0.0,1499.93,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3000.0,0.0,0.0
500QD00000WHiqKYAT,04892143,2025-04-01 14:07:37,2025-06-11 13:40:39,Technical Support,IPP Core,41130,72213,3,2,500QD00000WHiqKYAT,3109.43,0.27,3115.23,0.0,0.0,0.0,0.0,0.0,13607.85,0.0,10740.25,0.0,0.0
500QD00000YZyqZYAT,05040518,2025-05-14 14:52:13,2025-06-24 00:02:50,Technical Support,Ecom Core,3691,40150,3,2,500QD00000YZyqZYAT,3735.3,0.0,2091.22,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11401.27,0.0,0.0
500QD00000a7tZMYAY,05103612,2025-06-02 21:29:53,2025-10-03 13:47:46,Technical Support,IPP Adv. Payments,104501,125657,4,2,500QD00000a7tZMYAY,2472.7,0.0,4862.65,0.0,0.0,0.0,0.0,0.0,38822.53,0.0,6780.0,0.0,0.0
500QD00000dFhmGYAS,05279484,2025-07-25 12:09:32,2025-08-25 14:12:51,Technical Support,IPP Core,4382,29763,4,2,500QD00000dFhmGYAS,678.62,235.42,1385.33,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10423.97,0.0,0.0
500QD00000e621sYAA,05370998,2025-08-09 19:14:44,2025-08-27 23:52:08,Operational Support,Operational Support,2685,18352,3,2,500QD00000e621sYAA,2303.65,0.0,1546.03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3950.32,0.0,0.0


In [36]:
# 1. List the columns you want to combine
combine_cols = ["Handover", "In Progress", "New", "Waiting for 3rd Party"]

# 2. Add the new column to your existing final_report
# We use coalesce(col, 0) to treat missing/null values as 0 minutes
final_report = final_report.withColumn(
    "total_combined_elapsed_mins",
    F.round(
        sum(F.coalesce(F.col(c), F.lit(0)) for c in combine_cols), 
        2
    )
)

# 3. Verify that all original columns still exist
# final_report.printSchema()
final_report.limit(10).show()

500QD0000075aObYAI,03426305,2024-01-08 07:21:27,2024-01-09 21:27:09,Operational Support,Operational Support,2285,2285,2,0,500QD0000075aObYAI,0.0,0.0,1167.15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1167.15
500QD000007KOAJYA4,03436849,2024-01-10 12:59:29,2024-01-17 15:39:01,Operational Support,LATAM - Operational Support,10239,10239,1,0,500QD000007KOAJYA4,0.0,0.0,159.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3000.43,0.0,0.0,159.1
500QD000007by64YAA,03454088,2024-01-16 15:40:34,2024-06-21 12:44:29,Account Setup Operations,Account Setup Operations,225843,225843,1,0,500QD000007by64YAA,0.0,5.0,456.33,0.0,0.0,0.0,0.0,0.0,67162.6,0.0,0.0,0.0,0.0,67623.93
500QD000007xvsfYAA,03475201,2024-01-23 23:32:06,2024-03-13 23:41:26,Account Setup Operations,Account Setup Operations,51009,51009,1,0,500QD000007xvsfYAA,0.0,3877.17,124.33,0.0,0.0,0.0,0.0,0.0,7060.67,0.0,10537.83,0.0,0.0,11062.17
500QD0000086KaaYAE,03483161,2024-01-26 09:30:20,2024-03-18 11:59:14,Technical Support,None,75028,75028,1,0,500QD0000086KaaYAE,0.0,623.13,780.43,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20196.43,0.0,0.0,1403.56
500QD0000096yqkYAA,03533514,2024-02-12 14:28:03,2024-03-04 20:34:08,Account Setup Operations,Account Setup Operations,30606,30606,2,0,500QD0000096yqkYAA,2977.95,0.0,3936.18,0.0,0.0,0.0,0.0,0.0,2405.53,0.0,46.43,0.0,0.0,6341.71
500QD000009D96jYAC,03543863,2024-02-14 08:33:05,2024-03-06 02:02:51,Technical Support,Platforms,29849,29849,1,0,500QD000009D96jYAC,0.0,1295.4,4704.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3000.0,0.0,0.0,6000.0
500QD000009KU9oYAG,03550976,2024-02-16 03:23:56,2024-03-05 14:51:18,Disputes,Disputes,26607,26607,2,0,500QD000009KU9oYAG,1794.5,0.0,5576.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5576.8
500QD000009XZZpYAO,03561149,2024-02-20 16:33:22,2024-03-08 11:25:03,Technical Support,Ecom Cards,14081,18171,3,0,500QD000009XZZpYAO,3471.13,99.53,820.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3135.17,0.0,0.0,920.33
500QD000009c3MKYAY,03564771,2024-02-21 17:02:07,2024-03-14 13:29:45,Technical Support,Finance,21387,31467,1,0,500QD000009c3MKYAY,0.0,1126.48,2666.53,0.0,0.0,0.0,0.0,0.0,2489.58,0.0,3105.05,0.0,0.0,6282.59


In [37]:
final_report.filter("id = '500QD00000b8WDZYA2'").show()

500QD00000b8WDZYA2,05158954,2025-06-19 10:09:38,2026-01-09 09:23:55,Account Setup Operations,Japan - Operational Support,206714,206714,1,0,500QD00000b8WDZYA2,0.0,1800.0,1200.0,0.0,0.0,0.0,0.0,0.0,84000.0,0.0,0.0,0.0,0.0,87000.0


In [38]:
def get_monthly_report(df):
    """
    Groups closed tickets from 2025 onwards and calculates averages 
    for both the original elapsed time and the combined status measure.
    """
    
    monthly_stats = df.filter(
        (F.col("closed_date").isNotNull()) & 
        (F.col("closed_date").cast("timestamp") >= "2026-01-01") &
        (F.col("support_team").like('LATAM%') | F.col("support_team").like('Japan%'))
    ) \
    .withColumn(
        "month_period", 
        F.date_format(F.col("closed_date").cast("timestamp"), "yyyy-MM")
    ) \
    .groupBy("month_period","support_team") \
    .agg(
        F.countDistinct("id").alias("total_tickets"),
        # Original Average
        F.round(F.avg(F.col("actual_elapsed_time_in_mins")), 2)
            .alias("avg_actual_elapsed_time_in_mins"),
        # NEW: Average of your combined measure
        F.round(F.avg(F.col("total_combined_elapsed_mins")), 2)
            .alias("avg_total_combined_elapsed_mins")
    ) \
    .orderBy("month_period")

    return monthly_stats

In [39]:
monthly_summary = get_monthly_report(final_report)
monthly_summary.show()

2026-01,Japan - Disputes,140,1074.96,325.71
2026-01,LATAM - Technical Support,397,4203.58,1419.12
2026-01,LATAM - Operational Support,920,2214.06,718.47
2026-01,Japan - Operational Support,90,25847.9,10222.62
2026-01,Japan - Technical Support,73,4018.11,1576.83
2026-02,LATAM - Operational Support,1094,2063.12,832.8
2026-02,Japan - Disputes,123,1844.33,529.72
2026-02,Japan - Operational Support,97,16412.01,6747.69
2026-02,LATAM - Technical Support,419,3867.42,1407.09
2026-02,Japan - Technical Support,94,5468.19,2252.03
2026-03,Japan - Technical Support,92,9207.78,3830.95
